# Conexión y Consultas a Neo4j
He puesto en este notebook el código trasladado de un archivo python normal para trabajar con la base de datos Neo4j.

In [17]:

from neo4j import GraphDatabase


uri = "neo4j+s://9513c9ad.databases.neo4j.io"
username = "9513c9ad"
password = "rHZJ1U1LwZhbQdPItq2UfEoYzTScVv241N_85ZarT2E"

driver = GraphDatabase.driver(uri, auth=(username, password))

print("Conexión a Neo4j establecida correctamente")

Conexión a Neo4j establecida correctamente


## 2 Definir Neo4jConnection

In [18]:
class Neo4jConnection:

    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None

        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
        except Exception as e:
            print("Failed to create the driver:", e)

    def close(self):
        if self.__driver is not None:
            self.__driver.close()

    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None

        try:
            session = self.__driver.session(database=db) if db is not None else self.__driver.session()
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally:
            if session is not None:
                session.close()
        return response

## 3 Ejecutar la relación

In [19]:

conn = Neo4jConnection(uri=uri, user=username, pwd=password)


query = ''' MATCH (n), ()-[r]->()
            RETURN count(n) AS numNodos, count(r) AS numRelaciones'''

result = conn.query(query)
print("Resultado de la consulta:")
for record in result:
    print(record)

Resultado de la consulta:
<Record numNodos=5419536 numRelaciones=5419536>


## 4 Datos ID y propiedades de productos

In [20]:
conn = Neo4jConnection(uri=uri, user=username, pwd=password)

In [21]:
# Consulta de los cuatro primeros nodos Productos
query = """ MATCH (n:Product)
            RETURN n
            LIMIT 4 """

result = conn.query(query)
result
type(result[0])
type(result[0]["n"])

neo4j.graph.Node

In [22]:
# Presente los datos Id, label y properties de forma visual
for record in result:
    print('Id:', record["n"].element_id)
    print('Etiquetas:', record["n"].labels)
    print('Propiedades:', dict(record["n"]))
    print('')

Id: 4:d03cf31c-35f4-484d-abae-20fceb5d2175:89
Etiquetas: frozenset({'Product'})
Propiedades: {'unitPrice': 18.0, 'unitsInStock': 39, 'reorderLevel': 10, 'productID': '1', 'discontinued': False, 'productName': 'Chai', 'unitsOnOrder': 0}

Id: 4:d03cf31c-35f4-484d-abae-20fceb5d2175:90
Etiquetas: frozenset({'Product'})
Propiedades: {'unitPrice': 19.0, 'unitsInStock': 17, 'reorderLevel': 25, 'productID': '2', 'discontinued': False, 'productName': 'Chang', 'unitsOnOrder': 40}

Id: 4:d03cf31c-35f4-484d-abae-20fceb5d2175:91
Etiquetas: frozenset({'Product'})
Propiedades: {'unitPrice': 10.0, 'unitsInStock': 13, 'reorderLevel': 25, 'productID': '3', 'discontinued': False, 'productName': 'Aniseed Syrup', 'unitsOnOrder': 70}

Id: 4:d03cf31c-35f4-484d-abae-20fceb5d2175:92
Etiquetas: frozenset({'Product'})
Propiedades: {'unitPrice': 22.0, 'unitsInStock': 53, 'reorderLevel': 0, 'productID': '4', 'discontinued': False, 'productName': "Chef Anton's Cajun Seasoning", 'unitsOnOrder': 0}



In [23]:
# Consulta de los nodos Productos y de los nodos Supplier conectados por una relación SUPPLIES
query = """ MATCH (n:Product)<-[r:SUPPLIES]-(s:Supplier)
            RETURN n,r,s
            LIMIT 1 """

result = conn.query(query)
result

[<Record n=<Node element_id='4:d03cf31c-35f4-484d-abae-20fceb5d2175:89' labels=frozenset({'Product'}) properties={'unitPrice': 18.0, 'unitsInStock': 39, 'reorderLevel': 10, 'productID': '1', 'discontinued': False, 'productName': 'Chai', 'unitsOnOrder': 0}> r=<Relationship element_id='5:d03cf31c-35f4-484d-abae-20fceb5d2175:1152922604118474809' nodes=(<Node element_id='4:d03cf31c-35f4-484d-abae-20fceb5d2175:57' labels=frozenset({'Supplier'}) properties={'country': 'UK', 'supplierID': '1', 'contactTitle': 'Purchasing Manager', 'address': '49 Gilbert St.', 'city': 'London', 'phone': '(171) 555-2222', 'contactName': 'Charlotte Cooper', 'companyName': 'Exotic Liquids', 'postalCode': 'EC1 4SD', 'region': 'NULL', 'fax': 'NULL', 'homePage': 'NULL'}>, <Node element_id='4:d03cf31c-35f4-484d-abae-20fceb5d2175:89' labels=frozenset({'Product'}) properties={'unitPrice': 18.0, 'unitsInStock': 39, 'reorderLevel': 10, 'productID': '1', 'discontinued': False, 'productName': 'Chai', 'unitsOnOrder': 0}>) t

In [24]:
# Consulta de los Productos comprados por Clientes, y de sus Proveedores
query = """ MATCH path=(c:Customer)-[:PURCHASED]->()-[:ORDERS]->(:Product)<-[:SUPPLIES]-(:Supplier)
            WHERE c.companyName = 'Blauer See Delikatessen'
            RETURN path
            LIMIT 1 """

result = conn.query(query)
result

[<Record path=<Path start=<Node element_id='4:d03cf31c-35f4-484d-abae-20fceb5d2175:1009' labels=frozenset({'Customer'}) properties={'country': 'Germany', 'contactTitle': 'Sales Representative', 'address': 'Forsterstr. 57', 'city': 'Mannheim', 'phone': '0621-08460', 'contactName': 'Hanna Moos', 'companyName': 'Blauer See Delikatessen', 'postalCode': '68306', 'customerID': 'BLAUS', 'region': 'NULL', 'fax': '0621-08924'}> end=<Node element_id='4:d03cf31c-35f4-484d-abae-20fceb5d2175:81' labels=frozenset({'Supplier'}) properties={'country': 'Canada', 'supplierID': '25', 'contactTitle': 'Marketing Manager', 'address': '2960 Rue St. Laurent', 'city': 'Montréal', 'phone': '(514) 555-9022', 'contactName': 'Jean-Guy Lauzon', 'companyName': 'Ma Maison', 'postalCode': 'H1J 1C3', 'region': 'Québec', 'fax': 'NULL', 'homePage': 'NULL'}> size=3>>]

## 4 Cerrar la conexión

In [25]:

conn.close()
driver.close()

print("Conexiones cerradas correctamente")

Conexiones cerradas correctamente
